# 03 - Collaborative Filtering

This notebook demonstrates item-item collaborative filtering:
1. Build the user-item matrix
2. Compute cosine item-item similarities
3. Explore "movies similar to Toy Story"
4. Visualise similarity distributions
5. Generate top-N recommendations for a sample user

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

PROC_DIR = Path("../data/processed")
RAW_DIR = Path("../data/raw")
MODEL_DIR = Path("../models/collaborative")

## 1. Build the User-Item Matrix

In [ ]:
# Load processed data
ui_matrix = sp.load_npz(PROC_DIR / "user_item_matrix.npz")

with open(PROC_DIR / "movie_id_to_idx.json") as f:
    movie_id_to_idx = {int(k): int(v) for k, v in json.load(f).items()}
with open(PROC_DIR / "user_id_to_idx.json") as f:
    user_id_to_idx = {int(k): int(v) for k, v in json.load(f).items()}

idx_to_movie_id = {v: k for k, v in movie_id_to_idx.items()}
idx_to_user_id = {v: k for k, v in user_id_to_idx.items()}

# Load movie titles for display
movies_df = pd.read_csv(RAW_DIR / "movies.csv")
movie_titles = dict(zip(movies_df["movieId"], movies_df["title"]))

n_users, n_items = ui_matrix.shape
print(f"User-Item Matrix: {n_users:,} users x {n_items:,} items")
print(f"Non-zero entries: {ui_matrix.nnz:,}")
print(f"Sparsity: {1 - ui_matrix.nnz / (n_users * n_items):.4%}")

## 2. Compute Item-Item Cosine Similarities

We compute cosine similarity between item column vectors (users who rated both items).
To keep it tractable, we only store the top-K most similar items per movie.

In [ ]:
TOP_K = 50

# Item-item similarity on transposed matrix (items as rows)
# For demonstration, compute on a subset if the matrix is very large
item_matrix = ui_matrix.T.tocsr()  # items x users
print(f"Item matrix shape: {item_matrix.shape}")

# If pre-computed similarity exists, load it; otherwise compute a sample
sim_path = MODEL_DIR / "similarity.npz"
if sim_path.exists():
    print("Loading pre-computed similarity matrix ...")
    similarity = sp.load_npz(sim_path)
    print(f"Similarity matrix: {similarity.shape}, nnz={similarity.nnz:,}")
else:
    print("Computing item-item cosine similarity (sampling 2000 items for demo) ...")
    sample_size = min(2000, n_items)
    rng = np.random.RandomState(42)
    sample_idx = rng.choice(n_items, size=sample_size, replace=False)
    sample_matrix = item_matrix[sample_idx]
    sim_dense = cosine_similarity(sample_matrix)
    np.fill_diagonal(sim_dense, 0)
    similarity = sp.csr_matrix(sim_dense)
    print(f"Sample similarity: {similarity.shape}")

## 3. Movies Similar to Toy Story

Find the most similar movies to *Toy Story (1995)* using item-item CF.

In [ ]:
# Find Toy Story
toy_story_id = None
for mid, title in movie_titles.items():
    if "Toy Story" in title and "1995" in title:
        toy_story_id = mid
        break

if toy_story_id and toy_story_id in movie_id_to_idx:
    toy_story_idx = movie_id_to_idx[toy_story_id]
    print(f"Toy Story: movieId={toy_story_id}, idx={toy_story_idx}")
    print(f"Title: {movie_titles[toy_story_id]}")
    
    # Get similarity row
    if toy_story_idx < similarity.shape[0]:
        sim_row = similarity[toy_story_idx].toarray().flatten()
        top_indices = np.argsort(sim_row)[::-1][:15]
        
        print(f"\nTop 15 movies similar to Toy Story:")
        print("-" * 60)
        for rank, idx in enumerate(top_indices, 1):
            mid = idx_to_movie_id.get(idx, idx)
            title = movie_titles.get(mid, f"Unknown ({mid})")
            score = sim_row[idx]
            if score > 0:
                print(f"  {rank:2d}. {title:50s}  sim={score:.4f}")
    else:
        print("Toy Story index out of range for sample similarity. Load full model for accurate results.")
else:
    print("Toy Story not found in the filtered dataset.")

## 4. Similarity Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution of all non-zero similarity values
sim_values = similarity.data
axes[0].hist(sim_values, bins=50, color="steelblue", edgecolor="white", log=True)
axes[0].set_xlabel("Cosine Similarity")
axes[0].set_ylabel("Frequency (log)")
axes[0].set_title("Distribution of Non-zero Similarities")
axes[0].axvline(np.mean(sim_values), color="red", linestyle="--",
                label=f"Mean: {np.mean(sim_values):.3f}")
axes[0].legend()

# Number of non-zero neighbours per item
if sp.issparse(similarity):
    neighbours_per_item = np.diff(similarity.tocsr().indptr)
else:
    neighbours_per_item = (similarity > 0).sum(axis=1)
axes[1].hist(neighbours_per_item, bins=50, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Number of Similar Items")
axes[1].set_ylabel("Number of Items")
axes[1].set_title("Neighbours per Item")

# Max similarity per item
if sp.issparse(similarity):
    sim_csr = similarity.tocsr()
    max_sims = []
    for i in range(sim_csr.shape[0]):
        row = sim_csr[i].data
        max_sims.append(row.max() if len(row) > 0 else 0)
    max_sims = np.array(max_sims)
else:
    max_sims = similarity.max(axis=1)
axes[2].hist(max_sims[max_sims > 0], bins=50, color="steelblue", edgecolor="white")
axes[2].set_xlabel("Max Similarity Score")
axes[2].set_ylabel("Number of Items")
axes[2].set_title("Maximum Neighbour Similarity per Item")

plt.tight_layout()
plt.show()

print(f"Similarity stats:")
print(f"  Non-zero values: {len(sim_values):,}")
print(f"  Mean: {np.mean(sim_values):.4f}")
print(f"  Median: {np.median(sim_values):.4f}")
print(f"  Max: {np.max(sim_values):.4f}")

## 5. Generate Recommendations for a Sample User

Pick a user who has rated many movies and generate top-N recommendations.

In [ ]:
# Find a well-active user
user_activity = np.diff(ui_matrix.tocsr().indptr)
active_user_idx = np.argmax(user_activity)
active_user_id = idx_to_user_id[active_user_idx]

print(f"Sample user: {active_user_id} (index {active_user_idx})")
print(f"Number of ratings: {user_activity[active_user_idx]}")

# Get this user's rated movies
user_row = ui_matrix[active_user_idx].toarray().flatten()
rated_indices = np.where(user_row > 0)[0]

# Show top-rated movies by this user
user_ratings = [(idx, user_row[idx]) for idx in rated_indices]
user_ratings.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 10 highest-rated movies by this user:")
for idx, rating in user_ratings[:10]:
    mid = idx_to_movie_id.get(idx, idx)
    title = movie_titles.get(mid, f"Unknown ({mid})")
    print(f"  {title:50s}  rating={rating:.1f}")

In [ ]:
# Generate recommendations using weighted-sum approach
N = 20
seen_set = set(rated_indices)

# Score = sum of (similarity * rating) for items the user has rated
scores = similarity.T.dot(user_row)  # item scores
if hasattr(scores, 'A1'):
    scores = scores.A1  # convert matrix to array

# Zero out already-seen items
for idx in seen_set:
    if idx < len(scores):
        scores[idx] = 0

# Top-N
top_rec_indices = np.argsort(scores)[::-1][:N]

print(f"\nTop {N} Recommendations for User {active_user_id}:")
print("=" * 60)
for rank, idx in enumerate(top_rec_indices, 1):
    mid = idx_to_movie_id.get(idx, idx)
    title = movie_titles.get(mid, f"Unknown ({mid})")
    score = scores[idx]
    if score > 0:
        print(f"  {rank:2d}. {title:50s}  score={score:.3f}")

In [ ]:
# Visualise: user's rating distribution vs recommendation scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_row[user_row > 0], bins=9, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].set_title(f"User {active_user_id}'s Rating Distribution")

rec_scores = scores[top_rec_indices]
rec_scores = rec_scores[rec_scores > 0]
axes[1].barh(range(len(rec_scores)), rec_scores[::-1], color="steelblue", edgecolor="white")
axes[1].set_xlabel("Predicted Score")
axes[1].set_ylabel("Recommendation Rank")
axes[1].set_title(f"Top {N} Recommendation Scores")

plt.tight_layout()
plt.show()

print("Collaborative filtering exploration complete.")